# Chapter 8 — Linking Ontologies to Data
### Notebook 2 · Query rewriting

*Book reference: Section 8.3*

The other strategy: never build the graph. Translate the *query* instead, and let the database do what databases are good at. This is the notebook where the ontology disappears at run time.

In [1]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

{
 "mode": "offline (simulated LLM)",
 "chat_model": "claude-opus-5",
 "dspy_model": "anthropic/claude-opus-5",
 "fuseki": "in-memory rdflib",
 "artifacts": "C:\\Users\\marci\\OneDrive\\DEV\\EDU\\AIML\\Graph ML\\Ontology Engineering\\course\\artifacts"
}


In [2]:
import sys, json, logging
from pathlib import Path
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate)); break
sys.path.insert(0, str(Path.cwd()))

import ch08_toolkit as ch8
import pandas as pd
logging.getLogger("dspy").setLevel(logging.WARNING)

In [3]:
conn = ch8.build_database()

## 1. A query in the ontology's vocabulary

The queries are **conjunctive**: class atoms, property atoms and equality filters. That restriction is not for teaching convenience — rewriting is only *possible* for fragments like this, which is precisely why OWL 2 QL exists (Chapter 4 §4.2) and why §8.3 insists on a rewriting-friendly profile.

In [4]:
query = ch8.QUERIES['cardiac-patients-in-cardiology']
print('atoms:')
for s, p, o in query.property_atoms:
    print(f'  ?{s} {p.split("#")[-1]} ?{o}')
print('filters:', query.filters)
print('\nas SPARQL:')
print(ch8.to_sparql(query))

atoms:
  ?p hasDisorder ?d
  ?d category ?cat
  ?p inWard ?w
  ?w speciality ?spec
filters: [('cat', 'cardiac'), ('spec', 'cardiology')]

as SPARQL:
PREFIX med: <http://example.org/med#>
SELECT DISTINCT ?p WHERE {
  ?p med:hasDisorder ?d .
  ?d med:category ?cat .
  ?p med:inWard ?w .
  ?w med:speciality ?spec .
  FILTER(str(?cat) = "cardiac")
  FILTER(str(?spec) = "cardiology")
}


## 2. The rewriting

Each atom becomes the table its mapping names; a variable occurring in two atoms becomes a **join**; the IRI templates are rebuilt with string concatenation in the SELECT.

In [5]:
print(ch8.to_sql(query))

SELECT DISTINCT ('http://example.org/data/patient/' || p0.patient_id) AS p
FROM diagnosis p0, code_lookup p1, patient p2, ward p3
WHERE p0.patient_id = p2.id
  AND p0.code = p1.code
  AND p2.ward_id = p3.id
  AND p1.category = 'cardiac'
  AND p3.speciality = 'cardiology'


> **Look at what is not there.** No ontology, no triples, no reasoner — just a four-table join over the original schema. The ontology was a *compile-time* artefact: it decided which joins to write, and then it went away. That is the central idea of §8.3, and the reason rewriting scales to sources far too large to materialise.

## 3. The property that makes OBDA trustworthy

Two completely different execution paths. They must agree — and if they do not, the mapping is wrong. This check is cheap and it is the only one that catches the IRI/literal bug from Notebook 1.

In [6]:
graph = ch8.materialise(conn)
rows = []
for name, q in ch8.QUERIES.items():
    left = ch8.answers_via_materialisation(conn, q, graph=graph)
    right = ch8.answers_via_rewriting(conn, q)
    rows.append({'query': name, 'materialised': len(left),
                 'rewritten': len(right), 'agree': left == right})
print(pd.DataFrame(rows).to_string(index=False))
assert all(r['agree'] for r in rows)

                         query  materialised  rewritten  agree
                      patients             5          5   True
        patients-in-cardiology             3          3   True
patients-with-cardiac-disorder             3          3   True
cardiac-patients-in-cardiology             3          3   True


In [7]:
answers = ch8.answers_via_rewriting(conn, ch8.QUERIES['cardiac-patients-in-cardiology'])
print('cardiac patients in a cardiology ward:')
for row in answers:
    print('  ', row['p'])

cardiac patients in a cardiology ward:
   http://example.org/data/patient/101
   http://example.org/data/patient/103
   http://example.org/data/patient/104


## 4. Where the two strategies stop agreeing

They agree only while the materialisation is **fresh**. Update the source and one of them starts lying — without changing its behaviour in any visible way.

In [8]:
stale_graph = ch8.materialise(conn)         # snapshot taken now
conn.execute("INSERT INTO patient VALUES (106, 'Fatima', 1)")
conn.execute("INSERT INTO diagnosis VALUES (106, 'I21')")
conn.commit()
print('a new cardiac patient was just admitted to a cardiology ward\n')

q = ch8.QUERIES['cardiac-patients-in-cardiology']
stale = ch8.answers_via_materialisation(conn, q, graph=stale_graph)
live = ch8.answers_via_rewriting(conn, q)
print('stale materialisation says:', len(stale), 'patients')
print('query rewriting says      :', len(live), 'patients')
assert len(live) == len(stale) + 1

a new cardiac patient was just admitted to a cardiology ward

stale materialisation says: 3 patients
query rewriting says      : 4 patients


In [9]:
print('missing from the stale answer:',
      [r['p'] for r in live if r not in stale])
print('\nThe stale copy did not error, slow down, or warn. It returned a\n'
      'confident, well-formed, WRONG answer -- and in this scenario the\n'
      'missing row is a patient having a heart attack. Staleness is not a\n'
      'performance characteristic; it is a correctness one, which is why the\n'
      'MDP in Notebook 4 prices it as lost reward rather than added latency.')

missing from the stale answer: ['http://example.org/data/patient/106']

The stale copy did not error, slow down, or warn. It returned a
confident, well-formed, WRONG answer -- and in this scenario the
missing row is a patient having a heart attack. Staleness is not a
performance characteristic; it is a correctness one, which is why the
MDP in Notebook 4 prices it as lost reward rather than added latency.


In [10]:
refreshed = ch8.materialise(conn)
print('after re-materialising, the two agree again:',
      ch8.answers_via_materialisation(conn, q, graph=refreshed) == live)

after re-materialising, the two agree again: True


### Exercise 2.1 — Rewrite a query you write yourself

Write a conjunctive query for *wards that contain at least one respiratory patient*, and confirm both strategies agree.

> **Hint.** Three atoms, joined through the patient variable.

In [11]:
# YOUR CODE HERE


<details>
<summary>Solution 2.1</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [12]:
q = ch8.ConjunctiveQuery(
    select=['w'],
    property_atoms=[('p', ch8.MED + 'inWard', 'w'),
                    ('p', ch8.MED + 'hasDisorder', 'd'),
                    ('d', ch8.MED + 'category', 'cat')],
    filters=[('cat', 'respiratory')])
print(ch8.to_sql(q))
left = ch8.answers_via_materialisation(conn, q)
right = ch8.answers_via_rewriting(conn, q)
print('\nmaterialised:', left)
print('rewritten   :', right)
assert left == right and len(right) >= 1
print('\nNote the variable ?p appears in two atoms and becomes a join; ?w is\n'
      'projected through the ward IRI template.')

SELECT DISTINCT ('http://example.org/data/ward/' || p0.ward_id) AS w
FROM patient p0, diagnosis p1, code_lookup p2
WHERE p0.id = p1.patient_id
  AND p1.code = p2.code
  AND p2.category = 'respiratory'

materialised: [{'w': 'http://example.org/data/ward/2'}]
rewritten   : [{'w': 'http://example.org/data/ward/2'}]

Note the variable ?p appears in two atoms and becomes a join; ?w is
projected through the ward IRI template.


### Exercise 2.2 — Break a mapping and let the agreement check catch it

Corrupt one mapping so the two strategies disagree, and show the comparison detecting it. Explain why this check is worth running in CI.

In [13]:
# YOUR CODE HERE


<details>
<summary>Solution 2.2</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [14]:
wrong_table = [
    ch8.PropertyMap(p.predicate, p.table, p.subject_column, p.object_column,
                    p.subject_template,
                    None if p.predicate.endswith('hasDisorder') else p.object_template)
    for p in ch8.MAPPINGS['properties']]
broken = {'classes': ch8.MAPPINGS['classes'], 'properties': wrong_table}

q = ch8.QUERIES['patients-with-cardiac-disorder']
left = ch8.answers_via_materialisation(conn, q, mappings=broken)
right = ch8.answers_via_rewriting(conn, q, mappings=broken)
print('materialised:', len(left), 'rows')
print('rewritten   :', len(right), 'rows')
print('agree?', left == right)
assert left != right
print('\nThe two paths now disagree, so the bug is detectable without a gold\n'
      'answer, without a domain expert, and without anyone noticing the graph\n'
      'looked odd. That is exactly what makes it a CI check: it needs no oracle\n'
      'beyond the mapping itself.')

materialised: 0 rows
rewritten   : 4 rows
agree? False

The two paths now disagree, so the bug is detectable without a gold
answer, without a domain expert, and without anyone noticing the graph
looked odd. That is exactly what makes it a CI check: it needs no oracle
beyond the mapping itself.
